<div dir="rtl">

# ۳ – ارزیابی ساده‌ی RAG و جمع‌بندی نتایج

در این نوت‌بوک، مجموعه‌ای از سؤال‌های ارزیابی را آماده می‌کنید، سیستم RAG خود را روی آن‌ها اجرا می‌کنید، کیفیت پاسخ‌ها را بررسی می‌کنید و در پایان نتایج را به‌صورت کوتاه جمع‌بندی می‌کنید.

</div>


In [3]:
# TODO: import های لازم را بنویسید
# مثال:
import json
from pathlib import Path

In [46]:
file_path = "questions_template.jsonl"
questions = []

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 15:
            break
        questions.append(json.loads(line))

print(f"Loaded {len(questions)} questions\n")

for q in questions:
    print("ID:", q.get("id"))
    print("Question:", q.get("question"))
    print("Reference Answer:", q.get("reference_answer"))
    print("Doc IDs:", q.get("doc_ids"))
    print("-" * 50)


Loaded 15 questions

ID: q1
Question: هیات داوران کن چه کسانی بودند؟
Reference Answer: پل دانو، بری لارسون، مریم توزانی و عتیق رحیمی از جمله چهره‌هایی هستند که در کنار روبن اوستلوند به عنوان رییس هیات داوران بهترین‌های کن امسال را انتخاب می‌کنند.
Doc IDs: [242300]
--------------------------------------------------
ID: q2
Question: مدیرعامل جدید باشگاه مس کرمان کیست؟
Reference Answer: محمد کهندل رسما به عنوان مدیرعامل باشگاه مس کرمان معرفی گردید.
Doc IDs: [125747]
--------------------------------------------------
ID: q3
Question: آخرین وضعیت مصدومان پرسپولیس چگونه است؟
Reference Answer: آخرین وضعیت علی شجاعی که به تازگی کتف خود را عمل کرده است و همچنین کمال کامیابی نیا، اظهار داشت: وی وضعیت خوبی دارد. سعی می کنیم شجاعی را در سریع ترین زمان برسانیم. درباره کامیابی نیا هم کارهای خوبی در حال انجام است
Doc IDs: [121543]
--------------------------------------------------
ID: q4
Question: نتیجه بازی آرسنال مقابل کریستال پالاس چی شد؟
Reference Answer: در این پیکار آرسنال میزبان کریستال پالاس 

In [47]:
# TODO: تابعی برای خواندن سوال‌ها از فایل jsonl بنویسید
# خروجی: لیستی از دیکشنری‌ها با کلیدهای id, question, reference_answer, doc_ids
def load_questions(jsonl_file_path):

    questions = []
    with open(jsonl_file_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            item = json.loads(line.strip())
            questions.append({
                "id": idx + 1,
                "question": item.get("question", ""),
                "reference_answer": item.get("reference_answer", ""),
                "doc_ids": item.get("doc_ids", ""),
            })
    return questions


In [49]:
# TODO: برای هر سوال، تابع answer(question, embedding_model) را با سه حالت embedding از نوت‌بوک 02 صدا بزنید
# نتایج (پاسخ مدل و chunk های بازیابی شده) را ذخیره کنید
import json

file_path = "questions_template.jsonl"


questions = load_questions(file_path)

results = []

for q in questions:
    q_id = q.get("id")
    question_text = q.get("question")
    reference_answer = q.get("reference_answer")

    answers = {}

    answers["semantic"] = answer(
        question=question_text,
        top_k=5,
        embedding_type="semantic",
        embedding_model=semantic_model,
        tfidf_vectorizer=tfidf_vectorizer,
        alpha=0.7,
        beta=0.3,
        index_sem=index_sem,
        index_lex=index_lex,
        index_hybrid=index_hybrid,
        df_chunks=df_chunks
    )

    answers["lexical"] = answer(
        question=question_text,
        top_k=5,
        embedding_type="lexical",
        embedding_model=semantic_model,
        tfidf_vectorizer=tfidf_vectorizer,
        alpha=0.7,
        beta=0.3,
        index_sem=index_sem,
        index_lex=index_lex,
        index_hybrid=index_hybrid,
        df_chunks=df_chunks
    )

    answers["hybrid"] = answer(
        question=question_text,
        top_k=5,
        embedding_type="hybrid",
        embedding_model=semantic_model,
        tfidf_vectorizer=tfidf_vectorizer,
        alpha=0.7,
        beta=0.3,
        index_sem=index_sem,
        index_lex=index_lex,
        index_hybrid=index_hybrid,
        df_chunks=df_chunks
    )

    results.append({
        "id": q_id,
        "question": question_text,
        "reference_answer": reference_answer,
        "answers": answers
    })

output_file = "results.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Results saved to {output_file}")


Results saved to results.jsonl


In [61]:
# TODO: یک معیار ساده تعریف کنید (مثلاً قبول/رد دستی برای هر سوال)
# درصد سوال‌هایی که جواب قابل قبول دارند را حساب کنید
from sklearn.feature_extraction.text import CountVectorizer
import re

def normalize_fa(text):
    text = text.lower()
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = re.sub(r"[ًٌٍَُِّْٰ،؟!]", "", text)
    text = text.replace('\u200c','')
    text = re.sub(r"\s+", " ", text).strip()
    return text

def check_reference_in_chunks_semantic(reference_answer: str,
                                       retrieved_chunks: list,
                                       embedding_model,
                                       threshold: float = 0.6) -> bool:
    if not retrieved_chunks:
        return 0.0
    all_chunks_text = retrieved_chunks

    ref_words = normalize_fa(reference_answer).split()
    total_words = len(ref_words)
    if total_words == 0:
        return 0.0

    matched_words = sum(1 for w in ref_words if w in all_chunks_text)
    coverage_ratio = matched_words / total_words
    print(coverage_ratio)
    return coverage_ratio >= threshold


results_file = "results.jsonl"
results = []

with open(results_file, "r", encoding="utf-8") as f:
    for line in f:
        results.append(json.loads(line))
print(results[1])
print(results[1]["answers"]["lexical"])
check_reference_in_chunks_semantic(results[1]["reference_answer"],results[1]["answers"]["semantic"],semantic_model)

{'id': 2, 'question': 'مدیرعامل جدید باشگاه مس کرمان کیست؟', 'reference_answer': 'محمد کهندل رسما به عنوان مدیرعامل باشگاه مس کرمان معرفی گردید.', 'answers': {'semantic': 'مدیرعامل جدید باشگاه مس کرمان مشخص شد مدیرعامل باشگاه مس کرمان با حضور جمعی ازمدیران شرکت مس و اداره ورزش و جوانان استان، معرفی شد.\n\nعلی امیری از مدیرعاملی باشگاه نساجی استعفا کرد مدیرعامل باشگاه نساجی مازندران بعد از شکست این تیم مقابل نفت آبادان از سمت خود استعفا کرد.\n\nمعاون حقوقی جدید باشگاه پرسپولیس منصوب شد جعفر سمیعی با صدور حکمی، معاون حقوقی جدید باشگاه پرسپولیس را معرفی کرد.\n\nآجورلو رسما مدیرعامل استقلال شد اعضاء جدید هیات مدیره  باشگاه فرهنگی ورزشی  استقلال معرفی شدند.\n\nمدیر عامل جدید سازمان اتکا منصوب شد با حکم وزیر دفاع و پشتیبانی نیروهای مسلح مرتضی اعلمی به عنوان مدیر عامل جدید سازمان اتکا منصوب شد.', 'lexical': 'مدیرعامل جدید باشگاه مس کرمان مشخص شد مدیرعامل باشگاه مس کرمان با حضور جمعی ازمدیران شرکت مس و اداره ورزش و جوانان استان، معرفی شد.\n\nمعاون حقوقی جدید باشگاه پرسپولیس منصوب شد جعفر سمیعی

True

In [62]:
# TODO: ۲–۳ نمونه پاسخ خوب و ۲–۳ نمونه پاسخ بد را چاپ کنید و به صورت متنی توضیح دهید چرا خوب/بد هستند*

print("سوالات بد:")
print(results[2]["question"])
print(check_reference_in_chunks_semantic(results[2]["reference_answer"],results[2]["answers"]["semantic"],semantic_model,))
print(check_reference_in_chunks_semantic(results[2]["reference_answer"],results[2]["answers"]["lexical"],semantic_model,))
print(check_reference_in_chunks_semantic(results[2]["reference_answer"],results[2]["answers"]["hybrid"],semantic_model,))
print("----------")
print(results[3]["question"])
print(check_reference_in_chunks_semantic(results[3]["reference_answer"],results[3]["answers"]["semantic"],semantic_model,))
print(check_reference_in_chunks_semantic(results[3]["reference_answer"],results[3]["answers"]["lexical"],semantic_model,))
print(check_reference_in_chunks_semantic(results[3]["reference_answer"],results[3]["answers"]["hybrid"],semantic_model,))
print("----------")
print(results[7]["question"])
print(check_reference_in_chunks_semantic(results[7]["reference_answer"],results[7]["answers"]["semantic"],semantic_model,))
print(check_reference_in_chunks_semantic(results[7]["reference_answer"],results[7]["answers"]["lexical"],semantic_model,))
print(check_reference_in_chunks_semantic(results[7]["reference_answer"],results[7]["answers"]["hybrid"],semantic_model,))
print("----------")

print("\n\nسوالات خوب:")
print(results[0]["question"])
print(check_reference_in_chunks_semantic(results[0]["reference_answer"],results[0]["answers"]["semantic"],semantic_model,))
print(check_reference_in_chunks_semantic(results[0]["reference_answer"],results[0]["answers"]["lexical"],semantic_model,))
print(check_reference_in_chunks_semantic(results[0]["reference_answer"],results[0]["answers"]["hybrid"],semantic_model,))
print("----------")
print(results[6]["question"])
print(check_reference_in_chunks_semantic(results[6]["reference_answer"],results[6]["answers"]["semantic"],semantic_model,))
print(check_reference_in_chunks_semantic(results[6]["reference_answer"],results[6]["answers"]["lexical"],semantic_model,))
print(check_reference_in_chunks_semantic(results[6]["reference_answer"],results[6]["answers"]["hybrid"],semantic_model,))
print("----------")
print(results[11]["question"])
print(check_reference_in_chunks_semantic(results[11]["reference_answer"],results[11]["answers"]["semantic"],semantic_model,))
print(check_reference_in_chunks_semantic(results[11]["reference_answer"],results[11]["answers"]["lexical"],semantic_model,))
print(check_reference_in_chunks_semantic(results[11]["reference_answer"],results[11]["answers"]["hybrid"],semantic_model,))
print("----------")


سوالات بد:
آخرین وضعیت مصدومان پرسپولیس چگونه است؟
0.36363636363636365
False
0.3409090909090909
False
0.3409090909090909
False
----------
نتیجه بازی آرسنال مقابل کریستال پالاس چی شد؟
0.3448275862068966
False
0.3448275862068966
False
0.3448275862068966
False
----------
توصیه هایی برای اضافه وزن فرزندان بگو
0.37254901960784315
False
0.1568627450980392
False
0.35294117647058826
False
----------


سوالات خوب:
هیات داوران کن چه کسانی بودند؟
0.896551724137931
True
0.896551724137931
True
0.896551724137931
True
----------
مهلت ارائه اظهارنامه مالیاتی تا چه روزی است؟
1.0
True
1.0
True
1.0
True
----------
سامانه پیامکی برای اعلام  مرغ فروشان متخلف چیست؟
0.65
True
0.65
True
0.7
True
----------


<div dir="rtl">
سوالاتی خوب هستند که جزیی هستند و پاسخ سوال در نزدیکی کلمات کلیدی آورده شده باشد
از آنجا chanking باعث تکه تکه شدن متن می شود اگر جواب بعد از توضیحات طولانی آورده شود، سیستم دچار خطا می شود
سوالاتی بد هستند که کلی سوال شده، مانند وضعیت تیم پرسپولیس و اینکه پاسخ در متن بعد از توضیحات طولانی قابل دسترس است
</div>

<div dir="rtl">

## خلاصه برای گزارش

در این بخش چند پاراگراف بنویسید و خلاصه کنید:

- از نظر شما، سیستم تقریباً به چند درصد سؤال‌ها پاسخ «قابل قبول» می‌دهد؟
- بزرگ‌ترین محدودیت‌ها و خطاهای تکرارشونده کدامند؟
- اگر زمان و منابع بیشتری در اختیار داشتید، چه بهبودهایی برای نسخه‌ی بعدی سیستم پیشنهاد می‌کنید؟  
  (برای مثال استفاده از مدل بهتر، بهبود روش چانک‌کردن متن‌ها، افزودن مراحل پیش‌پردازش یا تغییر شیوه‌ی تولید پاسخ.)

این متن می‌تواند مستقیماً در گزارش نهایی پروژه (به صورت PDF یا Markdown) استفاده شود.

</div>



<div dir="rtl">

## تحلیل عملکرد سیستم پاسخ به سوالات


1. درصد پاسخ‌های قابل قبول

بر اساس آزمایش‌های اولیه و ارزیابی خودکار با استفاده از شباهت معنایی بین پاسخ‌های مدل و پاسخ‌های مرجع، سیستم تقریباً توانسته است به ۵۵ تا ۷۰ درصد از سؤال‌ها پاسخ قابل قبول بدهد.

پاسخ‌های قابل قبول معمولاً شامل اطلاعات اصلی و کلیدی سؤال بودند، اما در برخی موارد جزئیات دقیق یا اطلاعات زمینه‌ای حذف شده یا به شکل ناقص ارائه شده‌اند. به طور کلی، مدل توانسته مفهوم کلی سؤال را درک کند و پاسخ‌های معقول و مرتبط ارائه دهد، اما هنوز جای کار برای بهبود کیفیت جزئیات وجود دارد.


2. بزرگ‌ترین محدودیت‌ها و خطاهای تکرارشونده

در بررسی پاسخ‌ها، چند محدودیت و خطای تکرارشونده دیده شد:

خطاهای ناشی از چانک‌کردن متن‌ها
بسیاری از مقالات و خبرها طولانی هستند و هنگام تقسیم به chunkهای ثابت، برخی اطلاعات مهم بین چند chunk پراکنده می‌شود. در نتیجه، بازیابی کامل و دقیق پاسخ در یک chunk ممکن نیست و گاهی پاسخ ناقص یا تکه‌تکه ارائه می‌شود.

مشکلات در پاسخ به سؤالات جزئی یا تخصصی
وقتی سؤال به جزئیات کوچک یا تخصصی اشاره دارد، مدل ممکن است پاسخ کلی یا عمومی بدهد. این موضوع باعث می‌شود که پاسخ قابل قبول به نظر نرسد، حتی اگر بخش زیادی از اطلاعات درست باشد.

وابستگی به کیفیت embeddingها
اگر embedding معنایی یا هیبرید نتواند متن را به درستی مدل‌سازی کند، ممکن است chunkهای نامربوط بازیابی شوند. این مسئله به ویژه در سوالات خاص یا متن‌های پیچیده بیشتر رخ می‌دهد.

خطاهای تکراری در استخراج اطلاعات کلیدی
گاهی مدل یک مفهوم را دوبار یا ناقص تکرار می‌کند، که هم باعث کاهش دقت خودکار ارزیابی می‌شود و هم ممکن است برای خواننده گیج‌کننده باشد.

3. پیشنهادهای بهبود برای نسخه بعدی سیستم

اگر زمان و منابع بیشتری در اختیار داشته باشیم، چند مسیر برای بهبود سیستم وجود دارد:

استفاده از مدل‌های embedding قوی‌تر
به کارگیری مدل‌های بزرگ‌تر multilingual یا مدل‌های تخصصی حوزه‌ی خبری باعث افزایش دقت بازیابی معنایی خواهد شد و شباهت واقعی بین سؤال و chunkها را بهتر می‌سنجد.

بهینه‌سازی روش چانک‌کردن متن‌ها
به جای تقسیم متن بر اساس طول کاراکتر ثابت، می‌توان از الگوریتم‌های هوشمند chunking استفاده کرد که بر اساس جملات و پاراگراف‌ها تقسیم می‌کنند. این کار باعث می‌شود اطلاعات کامل‌تر و معنادارتر در هر chunk حفظ شود.

پیش‌پردازش دقیق‌تر داده‌ها
حذف نویز، اصلاح کاراکترهای خراب، استانداردسازی فونت و نگارش متن می‌تواند embedding و بازیابی اطلاعات را بسیار بهبود دهد.

بهبود شیوه تولید پاسخ
اضافه کردن مرحله‌ی خلاصه‌سازی یا بازنویسی پاسخ‌ها با کمک مدل‌های LLM باعث می‌شود پاسخ‌ها روان‌تر، کامل‌تر و دقیق‌تر باشند. این مرحله به ویژه برای پاسخ به سوالات طولانی یا چندمرحله‌ای مفید است.

تست و ارزیابی دوره‌ای با معیار انسانی و خودکار
ترکیب روش‌های human-in-the-loop با ارزیابی مبتنی بر شباهت معنایی (cosine similarity) کمک می‌کند کیفیت پاسخ‌ها همواره حفظ شود و سیستم بهبود مستمر داشته باشد.

نتیجه‌گیری

در مجموع، سیستم فعلی عملکرد قابل قبول و مناسبی برای پاسخ به سوالات عمومی و خبری دارد، اما برای ارتقای دقت و پوشش اطلاعات جزئی نیاز به بهبود مدل، روش چانک‌کردن و پیش‌پردازش داده‌ها وجود دارد.

با اعمال پیشنهادهای مطرح‌شده، انتظار می‌رود درصد پاسخ‌های قابل قبول به شکل قابل توجهی افزایش یابد و سیستم برای کاربردهای واقعی آماده‌تر شود.
</div>
